# 00 部署框架地图和选型

目标：把大模型部署相关框架按层次拆开，知道每类框架解决什么问题、什么时候该选它、面试时怎么讲清楚取舍。

整理时间：2026-05-25。部署生态变化很快，真实选型前要再看官方文档、版本支持、硬件和模型兼容性。


## 1. 框架不是一类东西

部署里常见的“框架”至少有四层：

```text
应用/API 层：FastAPI、OpenAI-compatible gateway、鉴权、限流、SSE
平台服务层：Triton、Ray Serve、KServe、BentoML、Kubernetes
LLM 推理引擎：vLLM、TGI、SGLang、TensorRT-LLM、LMDeploy、llama.cpp
优化组件：FlashAttention、FlashInfer、bitsandbytes、ONNX Runtime、torch.compile
```

面试里先把层次说清楚，比直接背框架名更重要。


In [ ]:
frameworks = [
    {
        "layer": "LLM serving engine",
        "name": "vLLM",
        "best_for": "通用 GPU 高吞吐文本/多模态服务",
        "learn_points": "PagedAttention、continuous batching、OpenAI API、prefix cache、speculative decoding、分布式推理",
        "when_to_choose": "需要快速把开源 LLM 做成高吞吐 OpenAI-compatible 服务",
        "watch_out": "不同模型/量化格式/硬件后端兼容性要实际压测",
        "docs": "https://docs.vllm.ai",
    },
    {
        "layer": "LLM serving engine",
        "name": "Hugging Face TGI",
        "best_for": "Hugging Face 生态生产部署",
        "learn_points": "Docker serving、streaming、metrics、tool/function calling、multi-backend",
        "when_to_choose": "团队模型都在 HF 生态，想要标准化容器服务",
        "watch_out": "和 vLLM/TensorRT-LLM 的性能差异要按模型和负载验证",
        "docs": "https://huggingface.co/docs/text-generation-inference",
    },
    {
        "layer": "LLM serving engine",
        "name": "SGLang",
        "best_for": "结构化输出、agent/workflow、低延迟服务",
        "learn_points": "structured generation、router、radix cache、speculative decoding、OpenAI API",
        "when_to_choose": "业务需要 JSON/工具调用/多步生成流程，并且追求吞吐和低延迟",
        "watch_out": "新特性多，生产前要固定版本并做兼容测试",
        "docs": "https://docs.sglang.ai",
    },
    {
        "layer": "LLM serving engine",
        "name": "TensorRT-LLM",
        "best_for": "NVIDIA GPU 上追求极致性能",
        "learn_points": "engine build、in-flight batching、paged KV cache、FP8/INT8/INT4、tensor parallel",
        "when_to_choose": "硬件主要是 NVIDIA，服务规模大，值得为性能做 engine 编译和调参",
        "watch_out": "工程复杂度和模型转换成本更高",
        "docs": "https://nvidia.github.io/TensorRT-LLM",
    },
    {
        "layer": "LLM serving engine",
        "name": "LMDeploy",
        "best_for": "InternLM/通义等中文生态和 TurboMind/PyTorch engine",
        "learn_points": "TurboMind、continuous batching、量化、多模型服务",
        "when_to_choose": "模型和团队生态接近 LMDeploy 支持范围",
        "watch_out": "不同 engine 对模型结构支持不同",
        "docs": "https://lmdeploy.readthedocs.io",
    },
    {
        "layer": "Local/edge runtime",
        "name": "llama.cpp",
        "best_for": "本地、CPU、边缘设备、GGUF 模型",
        "learn_points": "GGUF、CPU/GPU offload、llama-server、OpenAI-compatible endpoint",
        "when_to_choose": "没有大 GPU、需要本地隐私部署、Mac/CPU/边缘场景",
        "watch_out": "服务端调度能力和多租户能力通常不如数据中心 serving engine",
        "docs": "https://github.com/ggml-org/llama.cpp/blob/master/tools/server/README.md",
    },
    {
        "layer": "Local/edge runtime",
        "name": "Ollama",
        "best_for": "本地模型管理和快速体验",
        "learn_points": "模型拉取、Modelfile、本地 API、OpenAI compatibility",
        "when_to_choose": "个人开发、Demo、快速验证本地模型",
        "watch_out": "生产级多租户、调度和可观测性通常还要另建平台层",
        "docs": "https://docs.ollama.com/openai",
    },
    {
        "layer": "Compiler/runtime",
        "name": "MLC LLM",
        "best_for": "跨平台编译部署和移动/WebGPU 场景",
        "learn_points": "machine learning compiler、native deployment、WebGPU、REST serving",
        "when_to_choose": "目标是移动端、浏览器、非典型硬件或编译器路线",
        "watch_out": "模型转换和平台支持要提前验证",
        "docs": "https://llm.mlc.ai/docs/get_started/introduction.html",
    },
    {
        "layer": "Model serving platform",
        "name": "NVIDIA Triton Inference Server",
        "best_for": "多框架统一推理平台",
        "learn_points": "model repository、dynamic batching、ensemble、HTTP/gRPC、Prometheus、TensorRT-LLM backend",
        "when_to_choose": "服务不只 LLM，还有 CV/ASR/传统模型，需要统一平台治理",
        "watch_out": "LLM 高级能力通常要和 TensorRT-LLM/vLLM 等后端组合",
        "docs": "https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/index.html",
    },
    {
        "layer": "Model serving platform",
        "name": "Ray Serve",
        "best_for": "Python 分布式服务和 LLM 应用编排",
        "learn_points": "replica、autoscaling、deployment graph、Serve LLM、OpenAI API、vLLM integration",
        "when_to_choose": "需要 Python 复杂业务逻辑、分布式任务和模型服务放在一个体系里",
        "watch_out": "Ray 集群运维和资源隔离要掌握",
        "docs": "https://docs.ray.io/en/latest/serve/llm/index.html",
    },
    {
        "layer": "Kubernetes platform",
        "name": "KServe",
        "best_for": "Kubernetes 上标准化模型服务",
        "learn_points": "InferenceService、autoscaling、AI Gateway、OpenAI-compatible API、vLLM backend",
        "when_to_choose": "公司基础设施是 K8s，需要统一模型发布、弹性和网关治理",
        "watch_out": "学习成本偏平台工程，适合有 K8s 基础后再深入",
        "docs": "https://kserve.github.io/website/docs/model-serving/generative-inference/overview",
    },
    {
        "layer": "Model serving platform",
        "name": "BentoML",
        "best_for": "Python-first 模型 API、打包和部署",
        "learn_points": "service、Docker image、adaptive batching、OpenAI endpoint、vLLM backend",
        "when_to_choose": "想从 Python 项目快速封装可部署服务，同时保留工程可控性",
        "watch_out": "大规模 GPU 调度通常还要结合云/K8s 平台",
        "docs": "https://docs.bentoml.com/en/latest/examples/vllm.html",
    },
]

print("framework count:", len(frameworks))
print("layers:")
for layer in sorted({item["layer"] for item in frameworks}):
    print("-", layer)


## 2. 按层查看框架


In [ ]:
def print_table(rows, columns):
    widths = {col: max(len(col), *(len(str(row.get(col, ""))) for row in rows)) for col in columns}
    header = " | ".join(col.ljust(widths[col]) for col in columns)
    sep = "-+-".join("-" * widths[col] for col in columns)
    print(header)
    print(sep)
    for row in rows:
        print(" | ".join(str(row.get(col, "")).ljust(widths[col]) for col in columns))


for layer in sorted({item["layer"] for item in frameworks}):
    print("\n" + "=" * 100)
    print(layer)
    rows = [item for item in frameworks if item["layer"] == layer]
    print_table(rows, ["name", "best_for", "when_to_choose"])


## 3. 选型不是选最火，而是先约束场景

先问这几个问题：

- 硬件：NVIDIA GPU、AMD GPU、Intel CPU/GPU/NPU、Apple Silicon、纯 CPU？
- 目标：个人学习、Demo、公司内部 API、高并发生产、边缘端？
- 模型：Transformer 原生权重、GGUF、GPTQ/AWQ、FP8、MoE、多模态？
- 约束：低延迟、低成本、高吞吐、强隔离、K8s 原生、OpenAI-compatible？
- 团队能力：更熟 Python、K8s、NVIDIA 优化、Hugging Face 生态，还是 C++/边缘端？


In [ ]:
def recommend_stack(hardware="nvidia", scale="single_gpu", goal="learn", needs_k8s=False, local=False):
    hardware = hardware.lower()
    scale = scale.lower()
    goal = goal.lower()

    if local or hardware in {"cpu", "apple", "mac", "edge"}:
        return [
            "Ollama: 最快本地体验和模型管理",
            "llama.cpp: 学 GGUF、CPU/GPU offload、OpenAI-compatible 本地服务",
            "MLC LLM: 想研究移动端/WebGPU/编译器部署时再看",
        ]

    if needs_k8s:
        return [
            "KServe: K8s 原生模型服务和网关治理",
            "vLLM 或 SGLang: 作为 LLM runtime 后端",
            "Prometheus/Grafana/OpenTelemetry: 指标、日志和 trace",
        ]

    if hardware == "nvidia" and goal in {"max_perf", "latency", "production"}:
        return [
            "vLLM: 先做生产 baseline，最快形成 OpenAI-compatible 服务",
            "TensorRT-LLM: 追求极致性能时做专项优化",
            "Triton: 多模型、多框架统一服务和企业治理",
        ]

    if goal in {"structured", "agent", "json"}:
        return [
            "SGLang: 结构化输出、agent/workflow 和低延迟服务",
            "vLLM: 通用高吞吐 baseline",
            "Ray Serve: 复杂 Python 业务编排",
        ]

    return [
        "vLLM: 通用 LLM serving 第一优先级",
        "TGI: Hugging Face 生态标准容器服务",
        "BentoML 或 FastAPI: 做业务 API 包装",
    ]


scenarios = [
    dict(hardware="nvidia", scale="single_gpu", goal="learn"),
    dict(hardware="nvidia", scale="multi_gpu", goal="max_perf"),
    dict(hardware="cpu", local=True),
    dict(hardware="nvidia", goal="structured"),
    dict(hardware="nvidia", needs_k8s=True),
]

for scenario in scenarios:
    print("=" * 100)
    print("scenario:", scenario)
    for item in recommend_stack(**scenario):
        print("-", item)


## 4. 第一轮学习路线

建议按“能跑 -> 能测 -> 能优化 -> 能上线”的顺序学习：

1. Transformers `generate()`：理解模型最小推理链路。
2. vLLM：启动 OpenAI-compatible API，测 streaming、并发、TTFT、TPOT。
3. TGI 或 SGLang：对比另一套 serving engine 的功能和参数。
4. llama.cpp/Ollama：理解本地 GGUF 和边缘部署。
5. TensorRT-LLM：理解 engine build、in-flight batching、FP8/INT4、NVIDIA 专项优化。
6. Triton/Ray Serve/KServe/BentoML：理解服务平台层、弹性、监控、发布和治理。


In [ ]:
study_tasks = [
    ("vLLM", "用 Qwen2.5-0.5B/1.5B 启动 OpenAI API，写一个 streaming client，记录 TTFT/TPOT"),
    ("TGI", "用 Docker 启服务，比较参数、日志、metrics 和 vLLM 的差异"),
    ("SGLang", "做一个 JSON schema / structured output demo，观察 guided decoding 参数"),
    ("llama.cpp", "下载 GGUF，启动 llama-server，使用 OpenAI SDK 调本地模型"),
    ("TensorRT-LLM", "先读 engine build 流程和 benchmark 指标，再在 NVIDIA GPU 上做实验"),
    ("Ray Serve/KServe/BentoML", "把 vLLM 后端包装成可扩缩容、有 health check 和 metrics 的服务"),
]

for index, (name, task) in enumerate(study_tasks, start=1):
    print(f"{index}. {name}: {task}")


## 5. 面试回答模板

可以用这个结构回答“你了解哪些部署框架，怎么选？”：

> 我会先区分推理引擎和平台服务层。推理引擎里，vLLM 是通用高吞吐 baseline，TGI 更贴近 Hugging Face 生态，SGLang 适合结构化生成和 agent workflow，TensorRT-LLM 适合 NVIDIA 上极致性能，llama.cpp/Ollama 适合本地和边缘。平台层里，Triton 适合多框架统一推理，Ray Serve 适合 Python 分布式编排，KServe 适合 Kubernetes 标准化上线，BentoML 适合 Python-first 服务打包。最终选型要看硬件、模型格式、并发、延迟、团队运维能力和是否需要 OpenAI-compatible API。
